# Train Hate Speech Detection on Colab

This notebook clones or updates the project from GitHub, prepares data, then calls the production Python modules in `src/`.

Default branch is `main`. Change `BRANCH` in the first code cell if you want to train from another branch, for example `refactor`.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/thong7d/hate-speech-detection.git"
BRANCH = "main"
PROJECT_DIR = Path("/content/hate-speech-detection")

if PROJECT_DIR.exists():
    if not (PROJECT_DIR / ".git").exists():
        raise RuntimeError(f"{PROJECT_DIR} exists but is not a git repository. Remove it or choose another PROJECT_DIR.")
    os.chdir(PROJECT_DIR)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
    os.chdir(PROJECT_DIR)

print("Project directory:", Path.cwd())
subprocess.run(["git", "branch", "--show-current"], check=True)

In [ ]:
from pathlib import Path

skip_packages = ("underthesea", "gradio", "pyngrok", "google-generativeai")
source = Path("requirements.txt")
target = Path("requirements-colab.txt")
lines = []
for line in source.read_text(encoding="utf-8").splitlines():
    stripped = line.strip()
    if stripped and not stripped.startswith(skip_packages):
        lines.append(line)
target.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(target.read_text(encoding="utf-8"))

In [ ]:
!python -m pip install -U pip setuptools wheel
!python -m pip install -r requirements-colab.txt
!python -m pip install huggingface_hub sentencepiece protobuf
!python -m pip uninstall -y peft

In [ ]:
from pathlib import Path

from src.data.download import download_and_extract
from src.data.preprocessing import process_split

raw_paths = {
    "train": Path("data/raw/vihsd/train.csv"),
    "dev": Path("data/raw/vihsd/dev.csv"),
    "test": Path("data/raw/vihsd/test.csv"),
}
processed_paths = {
    "train": Path("data/processed/train.parquet"),
    "dev": Path("data/processed/dev.parquet"),
    "test": Path("data/processed/test.parquet"),
}

if not all(path.exists() for path in raw_paths.values()):
    download_and_extract("configs/paths.yaml")

for split in ("train", "dev", "test"):
    if not processed_paths[split].exists():
        df = process_split(raw_paths[split], processed_paths[split])
        print(f"{split}: {len(df)} rows -> {processed_paths[split]}")
    else:
        print(f"{split}: found {processed_paths[split]}")

In [ ]:
import os

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])
else:
    print("Khong du du lieu de xac minh HF_TOKEN hoac HF_TOKEN chua duoc cau hinh")

In [ ]:
!python -c "import torch, transformers, pandas, pyarrow, sklearn, src; print('OK')"

In [ ]:
!python -m src.training.train --config configs/train.yaml

In [ ]:
!python -m src.evaluation.evaluate --config configs/train.yaml

In [ ]:
push_flag = "--push-to-hub" if os.environ.get("HF_TOKEN") else ""
!python -m src.export.export_model --config configs/train.yaml {push_flag}

In [ ]:
!python -m src.evaluation.manual_tests --config configs/model.yaml

Outputs:

- Local artifact: `artifacts/hate_speech_model/`
- Final model: `artifacts/hate_speech_model/model/`
- Hugging Face repo: configured in `configs/train.yaml`